# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muska123-web/FlyRank-AI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import duckdb
print(duckdb.__version__)

1.3.2


In [3]:
!pip install -q duckdb duckdb-extension-httpfs


import duckdb
import os
from google.colab import userdata

# Read token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("hf_token")

con = duckdb.connect()

# Install/load HTTP extension
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    );
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 68.5 MB/s eta 0:00:00


In [4]:
!pip install -q huggingface_hub

from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=os.environ["HF_TOKEN"]
)

# Filter to just the fact_content_daily_performance table's files
fact_files = [f for f in files if "fact_content_daily_performance" in f]
for f in fact_files:
    print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [5]:
partition_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
columns = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{partition_path}')").df()
print(columns.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [6]:
partition_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
check = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content
    FROM read_parquet('{partition_path}')
""").df()

print(check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    n_rows   min_date   max_date  n_clients  n_content
0  9841378 2026-03-01 2026-03-31         55     331437


In [7]:
partition_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{partition_path}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []


In [8]:
partition_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

check = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content
    FROM read_parquet('{partition_path}')
""").df()

print(check)

    n_rows   min_date   max_date  n_clients  n_content
0  9841378 2026-03-01 2026-03-31         55     331437


In [9]:
dim_content_path = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# First, just see if this path pattern even resolves — dim tables are usually NOT month-partitioned
dim_columns = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{dim_content_path}')").df()
print(dim_columns.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [10]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=os.environ["HF_TOKEN"]
)

# Filter to just dim_content's files
dim_files = [f for f in files if "dim_content" in f]
for f in dim_files:
    print(f)

dim_content.parquet


In [11]:
api.list_repo_files

<bound method HfApi.list_repo_files of <huggingface_hub.hf_api.HfApi object at 0x7e4554f131d0>>

In [12]:
grain_check_content_only = con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS n
    FROM read_parquet('{dim_content_path}')
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(grain_check_content_only)

Empty DataFrame
Columns: [content_hash_id, n]
Index: []


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
join_check = con.sql(f"""
    SELECT
        f.report_date,
        f.content_hash_id,
        d.content_updated_date,
        f.report_date - d.content_updated_date AS days_since_update
    FROM read_parquet('{partition_path}') AS f
    LEFT JOIN read_parquet('{dim_content_path}') AS d
        ON f.content_hash_id = d.content_hash_id
    WHERE d.content_updated_date > f.report_date
    LIMIT 5
""").df()

print(join_check)

  report_date           content_hash_id content_updated_date  \
0  2026-03-01  content_b7e512995f79d5a6           2026-05-18   
1  2026-03-01  content_05597932fe4da067           2026-05-18   
2  2026-03-01  content_7a105f548d9c6916           2026-07-06   
3  2026-03-01  content_905aa32a0230694e           2026-05-18   
4  2026-03-01  content_a3ea9792f793ec72           2026-05-18   

   days_since_update  
0                -78  
1                -78  
2               -127  
3                -78  
4                -78  


In [14]:
created_check = con.sql(f"""
    SELECT
        f.report_date,
        f.content_hash_id,
        d.content_created_date,
        f.report_date - d.content_created_date AS days_since_created
    FROM read_parquet('{partition_path}') AS f
    LEFT JOIN read_parquet('{dim_content_path}') AS d
        ON f.content_hash_id = d.content_hash_id
    WHERE d.content_created_date > f.report_date
    LIMIT 5
""").df()

print(created_check)

  report_date           content_hash_id content_created_date  \
0  2026-03-02  content_8a7d1bf4d7918eca           2026-03-05   
1  2026-03-02  content_2397a834c7550738           2026-03-05   
2  2026-03-02  content_173b2b6fd93af8c1           2026-03-05   
3  2026-03-02  content_d75f1c2d7dc76814           2026-03-05   
4  2026-03-02  content_1289c797ccea2de7           2026-03-05   

   days_since_created  
0                  -3  
1                  -3  
2                  -3  
3                  -3  
4                  -3  


In [15]:
scope_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN d.content_created_date > f.report_date THEN 1 ELSE 0 END) AS future_created_rows,
        SUM(CASE WHEN d.content_created_date > f.report_date THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_affected
    FROM read_parquet('{partition_path}') AS f
    LEFT JOIN read_parquet('{dim_content_path}') AS d
        ON f.content_hash_id = d.content_hash_id
""").df()

print(scope_check)

   total_rows  future_created_rows  pct_affected
0     9841378              76562.0       0.77796


In [16]:
offset_check = con.sql(f"""
    SELECT
        MIN(f.report_date - d.content_created_date) AS min_days,
        MAX(f.report_date - d.content_created_date) AS max_days,
        AVG(f.report_date - d.content_created_date) AS avg_days
    FROM read_parquet('{partition_path}') AS f
    LEFT JOIN read_parquet('{dim_content_path}') AS d
        ON f.content_hash_id = d.content_hash_id
    WHERE d.content_created_date > f.report_date
""").df()

print(offset_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   min_days  max_days  avg_days
0        -8        -1 -2.651015


In [17]:
age_distribution = con.sql(f"""
    SELECT
        MIN(f.report_date - d.content_created_date) AS min_age,
        MEDIAN(f.report_date - d.content_created_date) AS median_age,
        AVG(f.report_date - d.content_created_date) AS avg_age,
        MAX(f.report_date - d.content_created_date) AS max_age,
        COUNT(*) AS n
    FROM read_parquet('{partition_path}') AS f
    LEFT JOIN read_parquet('{dim_content_path}') AS d
        ON f.content_hash_id = d.content_hash_id
    WHERE d.content_created_date IS NOT NULL
      AND d.content_created_date <= f.report_date
""").df()

age_distribution

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_age,median_age,avg_age,max_age,n
0,0,212.0,202.058918,494,9764816


In [18]:
age_buckets = con.sql(f"""
    SELECT
        CASE
            WHEN (f.report_date - d.content_created_date) <= 90
                THEN '0-90 days'
            WHEN (f.report_date - d.content_created_date) <= 180
                THEN '91-180 days'
            WHEN (f.report_date - d.content_created_date) <= 365
                THEN '181-365 days'
            ELSE '366+ days'
        END AS age_bucket,

        COUNT(*) AS n,

        AVG(f.gsc_impressions) AS avg_impressions,
        AVG(f.gsc_clicks) AS avg_clicks,
        AVG(
            CASE
                WHEN f.gsc_impressions > 0
                THEN 100.0 * f.gsc_clicks / f.gsc_impressions
            END
        ) AS avg_ctr

    FROM read_parquet('{partition_path}') AS f

    LEFT JOIN read_parquet('{dim_content_path}') AS d
        ON f.content_hash_id = d.content_hash_id

    WHERE d.content_created_date IS NOT NULL
      AND d.content_created_date <= f.report_date
      AND f.gsc_data_available IS TRUE

    GROUP BY age_bucket
    ORDER BY
        MIN(f.report_date - d.content_created_date)
""").df()

age_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_bucket,n,avg_impressions,avg_clicks,avg_ctr
0,0-90 days,1192154,74.754809,0.234974,0.336071
1,91-180 days,637836,90.264237,0.243987,0.268266
2,181-365 days,1385874,77.507641,0.224956,0.320931
3,366+ days,395185,67.180417,0.188069,0.242794


In [19]:
# MIXED — very old content (366+ days) has the lowest average impressions, clicks, and CTR, but performance does not decline consistently across all age buckets.

In [20]:
ctr_position_buckets = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN 'Position 1-3'
            WHEN gsc_avg_position <= 10 THEN 'Position 4-10'
            WHEN gsc_avg_position <= 20 THEN 'Position 11-20'
            ELSE 'Position 21+'
        END AS position_bucket,

        COUNT(*) AS n,

        AVG(gsc_avg_position) AS avg_position,

        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN 100.0 * gsc_clicks / gsc_impressions
            END
        ) AS avg_ctr

    FROM read_parquet('{partition_path}')

    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0

    GROUP BY position_bucket

    ORDER BY
        MIN(gsc_avg_position)
""").df()

ctr_position_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_position,avg_ctr
0,Position 1-3,564173,1.807196,0.491821
1,Position 4-10,1456122,6.059994,0.347264
2,Position 11-20,519223,14.330876,0.276991
3,Position 21+,908354,43.888638,0.128915


In [21]:
# CONFIRMED — average CTR decreased consistently as search position worsened, from 0.492% at positions 1–3 to 0.129% at positions 21+. This supports using position-specific CTR expectations rather than treating all low CTR values equally.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
ctr_thresholds = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN 'Position 1-3'
            WHEN gsc_avg_position <= 10 THEN 'Position 4-10'
            WHEN gsc_avg_position <= 20 THEN 'Position 11-20'
            ELSE 'Position 21+'
        END AS position_bucket,

        COUNT(*) AS n,

        MEDIAN(
            100.0 * gsc_clicks / gsc_impressions
        ) AS median_ctr,

        QUANTILE_CONT(
            100.0 * gsc_clicks / gsc_impressions, 0.25
        ) AS p25_ctr

    FROM read_parquet('{partition_path}')

    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0

    GROUP BY position_bucket

    ORDER BY MIN(gsc_avg_position)
""").df()

ctr_thresholds

,position_bucket,n,median_ctr,p25_ctr
0,Position 1-3,564173,0.0,0.0
1,Position 4-10,1456122,0.0,0.0
2,Position 11-20,519223,0.0,0.0
3,Position 21+,908354,0.0,0.0


In [24]:
impression_distribution = con.sql(f"""
    SELECT
        COUNT(*) AS n,
        MIN(gsc_impressions) AS min_impressions,
        MEDIAN(gsc_impressions) AS median_impressions,
        QUANTILE_CONT(gsc_impressions, 0.75) AS p75_impressions,
        QUANTILE_CONT(gsc_impressions, 0.90) AS p90_impressions,
        MAX(gsc_impressions) AS max_impressions

    FROM read_parquet('{partition_path}')

    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0
      AND gsc_avg_position <= 10
""").df()

impression_distribution

,n,min_impressions,median_impressions,p75_impressions,p90_impressions,max_impressions
0,2020295,1,23.0,83.0,227.0,40084


In [25]:
zero_click_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_high_volume,

        SUM(
            CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END
        ) AS n_zero_clicks,

        100.0 * SUM(
            CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END
        ) / COUNT(*) AS pct_zero_clicks

    FROM read_parquet('{partition_path}')

    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position > 0
      AND gsc_avg_position <= 10
      AND gsc_impressions >= 83
""").df()

zero_click_check

,n_high_volume,n_zero_clicks,pct_zero_clicks
0,507931,285336.0,56.176134


In [26]:
#Top-10 ranking
#AND
#≥ 83 impressions
# AND
#0 clicks
# Prioritize content for CTR review when it ranks in positions 1–10, has at least 83 impressions, and receives zero clicks. Rank qualifying observations by impression volume so the highest-exposure missed-click opportunities appear first.
#

In [27]:
baseline_queue = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_avg_position > 0
             AND gsc_avg_position <= 10
             AND gsc_impressions >= 83
             AND gsc_clicks = 0
            THEN gsc_impressions
            ELSE 0
        END AS score,

        CASE
            WHEN gsc_avg_position > 0
             AND gsc_avg_position <= 10
             AND gsc_impressions >= 83
             AND gsc_clicks = 0
            THEN 'high_visibility_zero_clicks'
            ELSE 'not_flagged'
        END AS reason_code,

        CASE
            WHEN gsc_avg_position > 0
             AND gsc_avg_position <= 10
             AND gsc_impressions >= 83
             AND gsc_clicks = 0
            THEN 'review_ctr'
            ELSE 'no_action'
        END AS action_label

    FROM read_parquet('{partition_path}')

    WHERE gsc_data_available IS TRUE

    ORDER BY score DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [28]:
baseline_queue.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action_label
0,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,37368,high_visibility_zero_clicks,review_ctr
1,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.181500,33383,high_visibility_zero_clicks,review_ctr
2,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.132532,32958,high_visibility_zero_clicks,review_ctr
3,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,31472,0,0.083407,31472,high_visibility_zero_clicks,review_ctr
4,2026-03-02,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28973,0,0.000311,28973,high_visibility_zero_clicks,review_ctr
5,2026-03-01,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28947,0,0.002245,28947,high_visibility_zero_clicks,review_ctr
6,2026-03-22,client_62f4a7e64f5e0096,content_34a70fea29d15f24,27410,0,3.129405,27410,high_visibility_zero_clicks,review_ctr
7,2026-03-03,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,24233,0,0.317996,24233,high_visibility_zero_clicks,review_ctr
8,2026-03-13,client_e547b89c05043229,content_757b1fa67827358d,19301,0,2.261644,19301,high_visibility_zero_clicks,review_ctr
9,2026-03-24,client_73cda7b4e4f265ea,content_fec55986a1868d62,17770,0,0.013056,17770,high_visibility_zero_clicks,review_ctr


In [29]:
position_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_positive_positions,

        SUM(
            CASE
                WHEN gsc_avg_position > 0
                 AND gsc_avg_position < 1
                THEN 1 ELSE 0
            END
        ) AS n_below_one,

        MIN(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS min_positive_position

    FROM read_parquet('{partition_path}')

    WHERE gsc_data_available IS TRUE
""").df()

position_check

,n_positive_positions,n_below_one,min_positive_position
0,3611061,101548.0,0.000311


In [33]:
baseline_queue = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_avg_position >= 1
             AND gsc_avg_position <= 10
             AND gsc_impressions >= 83
             AND gsc_clicks = 0
            THEN gsc_impressions
            ELSE 0
        END AS score,

        CASE
            WHEN gsc_avg_position >= 1
             AND gsc_avg_position <= 10
             AND gsc_impressions >= 83
             AND gsc_clicks = 0
            THEN 'high_visibility_zero_clicks'
            ELSE 'not_flagged'
        END AS reason_code,

        CASE
            WHEN gsc_avg_position >= 1
             AND gsc_avg_position <= 10
             AND gsc_impressions >= 83
             AND gsc_clicks = 0
            THEN 'review_ctr'
            ELSE 'no_action'
        END AS action_label

    FROM read_parquet('{partition_path}')

    WHERE gsc_data_available IS TRUE

    ORDER BY score DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [34]:
baseline_queue.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action_label
0,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,37368,high_visibility_zero_clicks,review_ctr
1,2026-03-22,client_62f4a7e64f5e0096,content_34a70fea29d15f24,27410,0,3.129405,27410,high_visibility_zero_clicks,review_ctr
2,2026-03-13,client_e547b89c05043229,content_757b1fa67827358d,19301,0,2.261644,19301,high_visibility_zero_clicks,review_ctr
3,2026-03-04,client_62f4a7e64f5e0096,content_0c5606abaaab3178,13827,0,4.112533,13827,high_visibility_zero_clicks,review_ctr
4,2026-03-29,client_a80fca3f171ed1de,content_046fc480045b88f5,13764,0,6.915650,13764,high_visibility_zero_clicks,review_ctr
5,2026-03-05,client_e547b89c05043229,content_69379902126ff53f,13726,0,2.532566,13726,high_visibility_zero_clicks,review_ctr
6,2026-03-29,client_23a62021009f63c4,content_bf078007df823490,13253,0,1.039915,13253,high_visibility_zero_clicks,review_ctr
7,2026-03-27,client_23a62021009f63c4,content_bf078007df823490,11952,0,1.063922,11952,high_visibility_zero_clicks,review_ctr
8,2026-03-04,client_62f4a7e64f5e0096,content_0bca6d9a85a9b408,11464,0,7.485694,11464,high_visibility_zero_clicks,review_ctr
9,2026-03-26,client_23a62021009f63c4,content_bf078007df823490,11440,0,1.328322,11440,high_visibility_zero_clicks,review_ctr


In [35]:
flagged_queue = baseline_queue[
    baseline_queue["action_label"] == "review_ctr"
].copy()

flagged_queue = flagged_queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

flagged_queue.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action_label
0,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,37368,high_visibility_zero_clicks,review_ctr
1,2026-03-22,client_62f4a7e64f5e0096,content_34a70fea29d15f24,27410,0,3.129405,27410,high_visibility_zero_clicks,review_ctr
2,2026-03-13,client_e547b89c05043229,content_757b1fa67827358d,19301,0,2.261644,19301,high_visibility_zero_clicks,review_ctr
3,2026-03-04,client_62f4a7e64f5e0096,content_0c5606abaaab3178,13827,0,4.112533,13827,high_visibility_zero_clicks,review_ctr
4,2026-03-29,client_a80fca3f171ed1de,content_046fc480045b88f5,13764,0,6.915650,13764,high_visibility_zero_clicks,review_ctr
5,2026-03-05,client_e547b89c05043229,content_69379902126ff53f,13726,0,2.532566,13726,high_visibility_zero_clicks,review_ctr
6,2026-03-29,client_23a62021009f63c4,content_bf078007df823490,13253,0,1.039915,13253,high_visibility_zero_clicks,review_ctr
7,2026-03-27,client_23a62021009f63c4,content_bf078007df823490,11952,0,1.063922,11952,high_visibility_zero_clicks,review_ctr
8,2026-03-04,client_62f4a7e64f5e0096,content_0bca6d9a85a9b408,11464,0,7.485694,11464,high_visibility_zero_clicks,review_ctr
9,2026-03-26,client_23a62021009f63c4,content_bf078007df823490,11440,0,1.328322,11440,high_visibility_zero_clicks,review_ctr


In [36]:
print("Number of flagged rows:", len(flagged_queue))

Number of flagged rows: 273374


In [37]:
import os

os.makedirs("work/outputs", exist_ok=True)

flagged_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", len(flagged_queue), "rows")

Saved: 273374 rows


In [38]:
from google.colab import files

files.download("work/outputs/baseline_action_score.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#1. review_ctr — 37,368 impressions at avg. position 8.61 but 0 clicks, making this a high-visibility zero-click opportunity. It could be wrong if the zero-click measurement is incomplete or anomalous rather
# than reflecting true search behavior.
# 2 review_ctr — 27,410 impressions at position 3.13 with 0 clicks. This is concerning because the content has strong search visibility but no recorded clicks. It could be wrong if the GSC click data for this observation is incomplete.
# 3 review_ctr — 19,301 impressions at position 2.26 with 0 clicks. It is flagged because it ranks very highly and receives substantial exposure without clicks. It could be wrong if the recorded impressions and clicks do not represent comparable or complete search activity.
# 4 review_ctr — 13,827 impressions at position 4.11 with 0 clicks. It qualifies as a high-visibility zero-click opportunity. It could be wrong if the zero clicks result from a data-quality or measurement issue rather than poor CTR.
# 5 review_ctr — 13,764 impressions at position 6.92 with 0 clicks. It is flagged because it receives substantial visibility within the top 10 but no clicks. It could be wrong if the observed zero clicks are incomplete or anomalous.
# 6 review_ctr — 13,726 impressions at position 2.53 with 0 clicks. Its high ranking and large impression count make the lack of clicks worth reviewing. It could be wrong if GSC did not capture the click activity completely.
# 7 review_ctr — 13,253 impressions at position 1.04 with 0 clicks. This is especially worth investigating because it is near position 1 while recording no clicks. It could be wrong if the position or click measurement for this observation is anomalous.
# 8 review_ctr — 11,952 impressions at position 1.06 with 0 clicks. It is flagged because near-position-1 visibility with zero clicks is unusual under our rule. It could be wrong if the underlying GSC measurements are incomplete or anomalous.
# 9 review_ctr — 11,464 impressions at position 7.49 with 0 clicks. It receives substantial first-page visibility without recorded clicks, so the rule recommends CTR review. It could be wrong if the zero-click observation is caused by incomplete measurement rather than a real CTR problem.
# 10 review_ctr — 11,440 impressions at position 1.33 with 0 clicks. Its high impression volume and very strong ranking make zero clicks worth investigating. It could be wrong if the GSC click or position values are anomalous or incomplete.

In [39]:
top10 = flagged_queue.head(10).copy()

top10[
    [
        "report_date",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action_label"
    ]
]

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action_label
0,2026-03-04,content_945d6ff91386c817,37368,0,8.613948,37368,high_visibility_zero_clicks,review_ctr
1,2026-03-22,content_34a70fea29d15f24,27410,0,3.129405,27410,high_visibility_zero_clicks,review_ctr
2,2026-03-13,content_757b1fa67827358d,19301,0,2.261644,19301,high_visibility_zero_clicks,review_ctr
3,2026-03-04,content_0c5606abaaab3178,13827,0,4.112533,13827,high_visibility_zero_clicks,review_ctr
4,2026-03-29,content_046fc480045b88f5,13764,0,6.915650,13764,high_visibility_zero_clicks,review_ctr
5,2026-03-05,content_69379902126ff53f,13726,0,2.532566,13726,high_visibility_zero_clicks,review_ctr
6,2026-03-29,content_bf078007df823490,13253,0,1.039915,13253,high_visibility_zero_clicks,review_ctr
7,2026-03-27,content_bf078007df823490,11952,0,1.063922,11952,high_visibility_zero_clicks,review_ctr
8,2026-03-04,content_0bca6d9a85a9b408,11464,0,7.485694,11464,high_visibility_zero_clicks,review_ctr
9,2026-03-26,content_bf078007df823490,11440,0,1.328322,11440,high_visibility_zero_clicks,review_ctr


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Threshold limitation: The 83-impression threshold comes from the 75th percentile of March 2026 top-10 observations.
# It is a transparent data-based threshold, but it may not generalize equally well to other months or clients with different traffic levels.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.